## **Installing and Importing Necessary Libraries and Dependencies**

In [1]:
!pip install -q \
    langchain \
    langchain-openai \
    langgraph \
    langsmith \
    langchain-community \
    langchain-huggingface \
    openai \
    faiss-cpu \
    sentence-transformers \
    pydantic \
    pandas \



In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import os
import re
import json
from typing import List, Optional, Literal, TypedDict, Annotated

import pandas as pd
from pydantic import BaseModel, Field

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langgraph.graph.message import add_messages

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

/var/folders/9p/1svmp3fn2m50m6k2pl_qj4rh0000gp/T/ipykernel_86665/1932942588.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
# Load the JSON file and extract values
file_name = 'config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary

    OPENAI_API_KEY = config.get("OPENAI_API_KEY")                               # Extract OpenAI API key
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract OpenAI base URL

    LANGCHAIN_TRACING_V2 = config.get("LANGCHAIN_TRACING_V2")                   # Extract LangSmith tracing flag
    LANGCHAIN_API_KEY = config.get("LANGCHAIN_API_KEY")                         # Extract LangSmith API key
    LANGCHAIN_PROJECT = config.get("LANGCHAIN_PROJECT")                         # Extract LangSmith project name


# Store OpenAI credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY                                   # Set OpenAI API key
os.environ['OPENAI_BASE_URL'] = OPENAI_API_BASE                                 # Set OpenAI API base URL


# Store LangSmith credentials in environment variables
os.environ['LANGCHAIN_TRACING_V2'] = LANGCHAIN_TRACING_V2                       # Enable LangSmith tracing
os.environ['LANGCHAIN_API_KEY'] = LANGCHAIN_API_KEY                             # Set LangSmith API key
os.environ['LANGCHAIN_PROJECT'] = LANGCHAIN_PROJECT   

For the problem at hand, we will use two LLMs to separate responsibilities
- a lightweight model for primary reasoning and generation, and
- a more complex model for evaluation and validation, ensuring better accuracy, reliability, and cost efficiency.

In [4]:
# Initialize LLM
llm_fast = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Initialize separate LLM for validation (more complex model)
llm_validator = ChatOpenAI(model="gpt-4o", temperature=0)

Initialize LangSmith client to enable tracing and evaluation of the agentic workflow, with automatic fallback if configuration is unavailable.

In [5]:
from langsmith import traceable, Client
from langsmith.evaluation import evaluate

# Initialize LangSmith client
try:
    langsmith_client = Client()
    LANGSMITH_ENABLED = True
    print("LangSmith tracing enabled")
except:
    LANGSMITH_ENABLED = False
    print("LangSmith not configured - tracing disabled")

LangSmith tracing enabled


## **Data Loading**

Loads the previous customer support ticket dataset for storage in the database.


In [6]:
file_path = "Datasets/ticket_data.csv"
df = pd.read_csv(file_path)

Combines the ticket subject and body into a single `text` column.

In [7]:
df["text"] = df["subject"].fillna("") + "\n" + df["body"].fillna("")

Displays the dataset shape, number of unique queues, and the first 3 rows for a quick data overview.


In [8]:
print(f"Shape: {df.shape}")
print(f"Unique queues: {df['queue'].nunique()}")
df.head(3)

Shape: (500, 8)
Unique queues: 10


,subject,body,answer,type,queue,priority,language,text
0,Reported Problem with Financial Data Integration,"Dear Customer Support, I am reporting an issue...","<name>, thank you for reporting the data integ...",Problem,General Inquiry,medium,en,Reported Problem with Financial Data Integrati...
1,NaN,Is it possible to receive comprehensive guidan...,I would be delighted to assist you in optimizi...,Request,Product Support,medium,en,\nIs it possible to receive comprehensive guid...
2,Request for Details on Adobe Sign HelloSign Su...,Can you please provide comprehensive details o...,<name> would be happy to provide information o...,Request,Billing and Payments,medium,en,Request for Details on Adobe Sign HelloSign Su...


### Valid Labels
Extracts the unique queues, ticket types, and priority values from the dataset.

In [9]:
VALID_QUEUES = sorted(df["queue"].dropna().unique().tolist())
VALID_TYPES = sorted(df["type"].dropna().unique().tolist())
VALID_PRIORITIES = sorted(df["priority"].dropna().unique().tolist())

print("Queues:", VALID_QUEUES)
print("Types:", VALID_TYPES)
print("Priorities:", VALID_PRIORITIES)

Queues: ['Billing and Payments', 'Customer Service', 'General Inquiry', 'Human Resources', 'IT Support', 'Product Support', 'Returns and Exchanges', 'Sales and Pre-Sales', 'Service Outages and Maintenance', 'Technical Support']
Types: ['Change', 'Incident', 'Problem', 'Request']
Priorities: ['high', 'low', 'medium']


## **Knowledge Base**

In this section, we build the knowledge base from previous ticket data and create storage to manage the evidence retrieved by the agent.

1. **Build Documents:** Create documents from the ticket context and resolution data.
2. **Create Vector Store:** Generate embeddings and store the documents in FAISS for semantic search.
3. **Store Evidence:** Create an evidence store to collect and manage retrieved documents.
4. **Format Evidence:** Format the retrieved evidence so it can be passed to the Response Agent.

Defines the embedding model and the path where the FAISS vector store will be saved.


In [10]:
EMBED_MODEL = "BAAI/bge-large-en-v1.5"
VECTORSTORE_PATH = "helpdesk_vectorstore"

Creates knowledge-base documents by combining each ticket’s subject, body, and resolution, while storing ticket metadata for later retrieval.


In [13]:
# Build LangChain Document objects from the knowledge base DataFrame
def build_kb_documents(df: pd.DataFrame) -> list[Document]:
    documents = []

    # Process each row in the DataFrame as one knowledge base entry
    for i, row in df.iterrows():
        # Get the resolution/answer and skip rows where it is missing
        answer = str(row.get("answer")).strip()

        if not answer or answer.lower() == "nan":
            continue

         # Extract the ticket subject and body
        subject = str(row.get("subject")).strip()
        body = str(row.get("body")).strip()

        # Combine subject and body to represent the original ticket context
        ticket_context = f"{subject}\n{body}".strip()

        # Combine the ticket context and its resolution
        # so both the problem and solution are available for semantic search
        page_content = f"Ticket: {ticket_context}\n\nResolution: {answer}"

        # Create a LangChain Document containing the content and metadata
        documents.append(Document(page_content=page_content, 
                                  metadata=
                                  {"kb_id": f"KB_{i}",
                                   # Store additional ticket attributes as metadata
                                    "queue": row.get("queue", ""),
                                    "type": row.get("type", ""),
                                    "priority": row.get("priority", ""),
                                    # Keep the clean answer separately for citation/display
                                    "answer_only": answer,
                                   }))
    return documents




Creates or loads the FAISS vector store by converting documents into embeddings and saving them for semantic search.


In [ ]:
# Create or load a FAISS vectorstore for semantic search
def get_vector_store(documents: list[Document], save_path: str) -> FAISS:

    # Initialize the embedding model used to convert text into vectors
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBED_MODEL,
    )
    # If a previously saved vectorstore exists, load it instead of rebuilding it
    if os.path.exists(save_path):
        print(f"Loading existing vectorstore from {save_path}")
        return FAISS.load_local(save_path, embeddings, allow_dangerous_deserialization=True)
    #  what does allow_dangerous_deserialization=True mean?
    #  It allows loading a FAISS index that may have been saved with potentially unsafe data.
    #  Use this with caution and only if you trust the source of the saved index.

    # If no saved vectorstore exists, create a new one from the documents
    print(f"Building new vectorstore with {len(documents)} documents")

    # Convert documents into embeddings and store them in FAISS
    vector_store = FAISS.from_documents(documents, embeddings)

    # Save the vectorstore locally so it can be reused later
    vector_store.save_local(save_path)

    # Return the newly created vectorstore
    return vector_store

In [14]:
docs = build_kb_documents(df)
vector_store = get_vector_store(docs, VECTORSTORE_PATH)
print(f"Knowledge base ready: {len(docs)} documents indexed")

/Users/m0p0rjw/IdeaProjects/AI/olympus/AI-HelpDesk/venev-Helpdesk/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1027 16:49:06.244000 86665 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 11500.99it/s]


Building new vectorstore with 500 documents
Knowledge base ready: 500 documents indexed


Creates a session-scoped evidence store to collect and deduplicate documents retrieved by the RAG Sub-Agent, which the Response Agent uses for grounded response generation.

In [15]:
# Stores the evidence documents retrieved during a single workflow run
class EvidenceStore:
    """Session-scoped store for Document objects, since tools return strings to the LLM."""

    def __init__(self):
        # Store the retrieved Document objects
        self.docs: List[Document] = []

        # Track KB IDs that have already been added
        # This prevents duplicate documents from being stored
        self._seen_ids: set = set()

    def add(self, new_docs: List[Document]):
        # Add newly retrieved documents to the evidence store
        for d in new_docs:

            # Get the unique KB ID from the document metadata
            kb_id = d.metadata.get("kb_id")

            # Only add the document if it has not already been stored
            if kb_id not in self._seen_ids:
                self.docs.append(d)
                self._seen_ids.add(kb_id)

    def all(self) -> List[Document]:
        # Return all documents currently stored as evidence
        return list(self.docs)

    def clear(self):
        # Reset the evidence store
        self.docs = []
        self._seen_ids = set()

    def summary(self) -> str:
        # Return a message when no evidence has been collected
        if not self.docs:
            return "No evidence gathered yet."

        # Return the number of documents and their KB IDs
        return f"{len(self.docs)} documents in evidence store. IDs: {[d.metadata.get('kb_id') for d in self.docs]}"


# Create one shared evidence store for each workflow run
# This store should be reset at the beginning of every new ticket
evidence_store = EvidenceStore()

Formats the retrieved evidence into a clean, size-limited context for the Response Agent.


In [ ]:
def format_evidence(docs: List[Document], max_chars : int) -> str:
    parts, total = [], 0

    for d in docs: 
        content = d.metadata.get("answer_only", d.page_content)

        if total + len(content) > max_chars:
            break
        parts.append(content)
        total += len(content)

    return "\n".join(parts) if parts else "No evidence available."

## **Tools: Orchestrator Agent**

In this section, we define the tools used by the Orchestrator Agent to manage the ticket workflow.

* **Intake Tool:** Parses the ticket and extracts clean ticket.
* **Triage Tool:** Classifies the queue, type, priority, and confidence.
* **Escalation Tool:** Routes low-confidence or failed cases to a human queue.
* **Delivery Tool:** Sends the validated response to the customer.

### Intake Tool
Normalizes the incoming ticket text and records that the ticket has been received.

In [16]:
@traceable(name="intake_tool")
def intake_tool(ticket: str):

    """ Intake a support tickt and normalize the information 
    """
    ticket = ticket.strip()
    # Further normalization logic can be added here
    return {
        "ticket_text": ticket,
        "reasoning_trail":["✓ Intake: Ticket received and normalized"]
    }

### Triage Tool
Defines an LLM-based triage tool that classifies tickets into queue, type, priority, confidence, and reasoning using a structured output.

In [17]:
class TriageResult(BaseModel)   :
    queue: str
    type: str
    priority: str
    confidence: float
    reasoning: str

In [ ]:
TRIAGE_SYSTEM_PROMPT = """ You are a Expert support ticket classifier. 
Your taks is identifying the appropriate queue, type, and priority for incoming support tickets based on their content.
the Valid values for each field are as follows:
- queue: {VALID_QUEUES}
- type: {VALID_TYPES}
- priority: {VALID_PRIORITIES}
- confidence: Your confidence level in the classification : 0.0(very low) to 1.0(very high).
- reasoning: Your reasoning behind the classification.

IMPORTANT POINTS: 
- Always provide a confidence score for your classification.
- Ensure your reasoning is clear and justifies your classification.
- Use only the valid values specified for each field.
- If unsure, choose the most closet option based on the ticket content.
- If the ticket is vague or could not match any requirements, use lower confidence

Return JSON matching the schema.
"""

triage_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", TRIAGE_SYSTEM_PROMPT ),
        ("human", "Ticket: {TICKET_CONTENT}")
    ]
)

triage_chain = triage_prompt | llm_fast.with_structured_output(TriageResult)


In [22]:
@traceable(name = "triage_tool")
def triage_tool(ticket: str) -> dict :
    return triage_chain.invoke({"TICKET_CONTENT": ticket}).model_dump()

### Escalation Tool
Escalates the ticket to a human agent with the specified reason and triage details.

In [23]:
@traceable("escalation_tool")
def escalation_tool(reason:str, triage: dict) -> dict:
        print(f"[ESCALATE] queue={triage.get('queue')} reason={reason}")
        return {
                "final_response": f"Escalated to human agent due to reason: {reason}",
                "escalated": True,
                "escalated_reason": reason
        }

### Delivery Tool
Acts as the final node in the orchestration flow, passing through and returning the final response to the customer.


In [25]:
@traceable("delivery_tool")
def delivery_tool(response:str) -> dict:
        print(f"[DELIVER] Response length = {len(response)} chars")
        return {
                "final_response": response,
                "escalated": False
        }

## **Tools: RAG Sub-Agent**

The RAG Sub-Agent uses these tools to search the knowledge base and gather sufficient evidence for the support response.

* **Query Formulation Tool:** Generates 3 targeted search queries from the support ticket.
* **Vector Search Tool:** Searches the FAISS knowledge base and retrieves relevant evidence.
* **Sufficiency Check Tool:** Evaluates whether the retrieved evidence is sufficient to answer the ticket.

### Query Formulation Tool
An LLM-based tool that generates three targeted search queries from the support ticket to retrieve relevant knowledge base evidence.


In [26]:
class QueryPlan(BaseModel):
    queries: List[str] = Field(..., description="Exactly 3 distinct queries", min_length=3, max_length=3)

In [ ]:
QUERY_FORMULATION_SYSTEM_PROMPT= f""" 
You are planning to search the knowledge base for relevant information based on user tickets.
Rules: 
1. GENERATE EXACTLY 3 distinct non-overlapping short queries(Not complete sentences)
2. Focus On : product names, error codes , symptoms , processes 
3. Each query should cover a different angle (e.g. one for the symptom, one for the cause, one for the fix)
- If a rationale is provided from a prior failed retrieval, use it to target the missing information

Return JSON matching the schema
"""

QUERY_FORMULATION_PROMPT = ChatPromptTemplate.from_messages([
    ("system", QUERY_FORMULATION_SYSTEM_PROMPT),
    ("human", "Ticket : {ticket}, \n\ncontext/prior gap rationale : {rationale}")
    ])

query_chain = QUERY_FORMULATION_PROMPT | llm_fast.with_structured_output(QueryPlan)

In [32]:
@tool
@traceable(name = "query_formulation_tool")
def query_formulation_tool(ticket: str, rationale: str) -> str:
    """Generate exactly 3 distinct search queries for the knowledge base.
    Optionally accepts a rationale from a prior failed retrieval to target missing info.
    Returns the queries as a numbered list."""
    plan =  query_chain.invoke({"ticket": ticket, "rationale": rationale or "None"})
    return "\n".join(f"{i+1}. {q}" for i, q in enumerate(plan.queries))

### Vector Search Tool
The Vector Search Tool retrieves relevant evidence from the FAISS knowledge base for the queries generated by the RAG Sub-Agent.

1. **Parse Queries:** Extracts individual queries from the input and removes numbering.
2. **Search FAISS:** Runs each query once against the vector store to find similar documents.
3. **Deduplicate Results:** Removes duplicate documents using their `kb_id`.
4. **Store Evidence:** Adds the unique retrieved documents to the shared evidence store.
5. **Return Metadata:** Returns the retrieved document IDs, queues, similarity scores, and evidence-store summary for the agent.

In [33]:
# @tool
@traceable(name="vector_search_tool")
def vector_search_tool(queries: str, k: int = 4) -> str:
    """Run vector search against the KB using newline-separated queries."""

    # 1. Parse Queries:
    # Extract individual queries and remove numbering such as "1. " or "2) "
    query_list = []
    for line in queries.strip().split("\n"):
        line = line.strip()
        if not line:
            continue

        line = re.sub(r"^\d+[\.\)]\s*", "", line)
        query_list.append(line)

    if not query_list:
        return "No queries provided."

    new_docs = []
    metadata_log = []
    seen_this_call = set()

    # 2. Search FAISS:
    # Run each query once against the FAISS vector store
    for q in query_list:
        results = vector_store.similarity_search_with_score(q, k=k)

        for doc, score in results:
            kb_id = doc.metadata.get("kb_id")

            # 3. Deduplicate Results:
            # Skip documents that were already retrieved during this call
            if kb_id in seen_this_call:
                continue

            seen_this_call.add(kb_id)         

            new_docs.append(doc) 

            # Capture metadata for the agent to reason about the results
            metadata_log.append({
                "query": q[:40],
                "kb_id": kb_id,
                "queue": doc.metadata.get("queue"),
                "score": round(float(score), 3),
            })

    # 4. Store Evidence:
    # Add the unique retrieved documents to the shared evidence store
    evidence_store.add(new_docs)

    # 5. Return Metadata:
    # Return retrieved IDs, queues, similarity scores,
    # and the current evidence-store summary
    lines = [
        f"Retrieved {len(new_docs)} new unique docs "
        f"across {len(query_list)} queries."
    ]

    for m in metadata_log[:10]:
        lines.append(f"  {m}")

    lines.append(f"\nEvidence store now: {evidence_store.summary()}")

    return "\n".join(lines)

### Sufficiency Check Tool
An LLM-based tool that evaluates whether the retrieved evidence is sufficient to answer the support ticket and identifies any missing information.


In [34]:
class Sufficiency(BaseModel):
    sufficient: bool
    missing_info: List[str] =  Field(default_factory=list)
    rationale: str